# 04_00 — Avaliação do Modelo por Voo

**Input:** artefatos de `06_models/` (isolation_forest, scaler, selected_features)  
**Input:** voos de `04_feature/` (feature-engineered)  

Este notebook aplica o modelo treinado pelo pipeline **individualmente em cada voo**,
permitindo visualizar:
- O anomaly score ao longo do tempo
- O momento real da falha vs. o primeiro alerta do modelo
- A latência de detecção por voo
- Uma tabela-resumo comparando todos os voos

## Imports e caminhos

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import yaml

from aeroespacial_2.pipelines.model_training.nodes import create_windows

ROOT       = Path("..")
MODELS_DIR = ROOT / "data/06_models"
FEAT_DIR   = ROOT / "data/04_feature"

_params = yaml.safe_load((ROOT / "conf/base/parameters.yml").read_text())["model_training"]

TARGET_COL    = "target_fault"
TIMESTAMP_COL = "timestamp"
WINDOW_SIZE   = _params["window_size"]
SKIP_SECONDS  = _params.get("skip_seconds", 0.0)

print(f"skip_seconds carregado de parameters.yml: {SKIP_SECONDS} s")

## Carrega artefatos do modelo

In [ ]:
model           = joblib.load(MODELS_DIR / "isolation_forest.pkl")
scaler          = joblib.load(MODELS_DIR / "feature_scaler.pkl")
selected_features = json.loads((MODELS_DIR / "selected_features.json").read_text())

print(f"Features selecionadas ({len(selected_features)}):")
for i, f in enumerate(selected_features, 1):
    print(f"  {i:2d}. {f}")

## Função de avaliação por voo

In [ ]:
def evaluate_flight(df: pd.DataFrame, flight_name: str) -> dict | None:
    """Aplica o modelo em um único voo e retorna métricas."""
    missing = [f for f in selected_features if f not in df.columns]
    if missing:
        print(f"  [SKIP] {flight_name} — features ausentes: {missing}")
        return None

    if TARGET_COL not in df.columns:
        df = df.copy()
        df[TARGET_COL] = 0

    # Remove fase de transição inicial: aeronave desce de altitude acima do cruise
    # para ~50 m nos primeiros segundos — valores de energia/altitude são similares
    # aos da falha e causam falsos alarmes. O skip é aplicado aqui na avaliação
    # e também no treino (via build_training_data).
    if SKIP_SECONDS > 0.0:
        df = df[df[TIMESTAMP_COL] >= SKIP_SECONDS].copy()

    if len(df) <= WINDOW_SIZE:
        print(f"  [SKIP] {flight_name} — muito curto após skip ({len(df)} linhas)")
        return None

    X, y = create_windows(df, WINDOW_SIZE, selected_features, TARGET_COL)
    timestamps = df[TIMESTAMP_COL].values[WINDOW_SIZE:]

    X_scaled = scaler.transform(X)
    raw_preds = model.predict(X_scaled)
    y_pred    = np.where(raw_preds == -1, 1, 0)
    scores    = model.decision_function(X_scaled)

    fault_indices = np.where(y == 1)[0]
    has_fault = len(fault_indices) > 0

    result = {
        "flight":       flight_name,
        "has_fault":    has_fault,
        "n_samples":    len(y),
        "timestamps":   timestamps,
        "y_true":       y,
        "y_pred":       y_pred,
        "scores":       scores,
        "fault_time":   None,
        "detect_time":  None,
        "latency_s":    None,
        "false_alarms": int((y_pred[y == 0] == 1).sum()),
    }

    if has_fault:
        real_idx   = fault_indices[0]
        fault_time = timestamps[real_idx]
        post_fault = np.where((y_pred == 1) & (np.arange(len(y_pred)) >= real_idx))[0]
        result["fault_time"] = float(fault_time)
        if len(post_fault) > 0:
            detect_time = timestamps[post_fault[0]]
            result["detect_time"] = float(detect_time)
            result["latency_s"]   = float(detect_time - fault_time)

    return result

## Fase de transição inicial

Os voos são gravados com a aeronave já no ar, mas frequentemente acima da altitude de cruise (~50 m).
Durante a descida inicial até o cruise, as features de energia e altitude assumem valores similares
aos da falha de motor — causando falsos alarmes no início de muitos voos.

A tabela abaixo mostra o `pos_z_local` no início e no cruise de cada voo, e até quando dura a transição.
O parâmetro `skip_seconds` (lido de `parameters.yml`) exclui este período da avaliação e do treino.

In [ ]:
print(f"skip_seconds = {SKIP_SECONDS} s  (de conf/base/parameters.yml)\n")

rows_z = []
for path in sorted(FEAT_DIR.glob("*.csv")):
    df = pd.read_csv(path)
    if "pos_z_local" not in df.columns:
        continue
    t_max = df[TIMESTAMP_COL].max()
    z_inicio  = df.loc[df[TIMESTAMP_COL] < 5,  "pos_z_local"].mean()
    z_cruise  = df.loc[(df[TIMESTAMP_COL] >= 20) & (df[TIMESTAMP_COL] < t_max - 20), "pos_z_local"].mean()
    delta = abs(z_inicio - z_cruise) if not pd.isna(z_cruise) else float("nan")
    # primeiro instante em que pos_z fica dentro de ±3 m do cruise
    if not pd.isna(z_cruise) and delta > 3:
        stable_mask = (df["pos_z_local"] - z_cruise).abs() < 3.0
        idx = stable_mask.idxmax() if stable_mask.any() else None
        t_stable = float(df.loc[idx, TIMESTAMP_COL]) if idx is not None else float("nan")
    else:
        t_stable = 0.0
    rows_z.append({
        "Voo": path.stem.replace("carbonZ_", ""),
        "z_início (m)": round(z_inicio, 1),
        "z_cruise (m)": round(z_cruise, 1) if not pd.isna(z_cruise) else "—",
        "t_estável (s)": round(t_stable, 1),
        "skip cobre?": "✓" if t_stable <= SKIP_SECONDS else f"✗ ({t_stable:.1f}s)",
    })

df_z = pd.DataFrame(rows_z)
print(f"Transições que excedem skip_seconds={SKIP_SECONDS}s:")
problemas = df_z[df_z["skip cobre?"].str.startswith("✗")]
print(problemas[["Voo", "z_início (m)", "z_cruise (m)", "t_estável (s)"]].to_string(index=False) if len(problemas) else "  Nenhuma — skip cobre todas as transições.")
print()
df_z

## Avalia todos os voos

In [ ]:
all_results = []
csv_files   = sorted(FEAT_DIR.glob("*.csv"))

print(f"Voos encontrados: {len(csv_files)}\n")
for path in csv_files:
    df   = pd.read_csv(path)
    name = path.stem
    res  = evaluate_flight(df, name)
    if res is not None:
        all_results.append(res)
        tag = "✓ DETECTADO" if res["detect_time"] else ("✗ NÃO DETECTADO" if res["has_fault"] else "— sem falha")
        lat = f"latência={res['latency_s']:.2f}s" if res["latency_s"] is not None else ""
        print(f"  {name[:60]:<60} {tag}  {lat}")

print(f"\nTotal avaliado: {len(all_results)} voos")

## Tabela-resumo

In [ ]:
rows = []
for r in all_results:
    rows.append({
        "Voo":           r["flight"].replace("carbonZ_", ""),
        "Falha real":    "Sim" if r["has_fault"] else "Não",
        "Detectado":     "Sim" if r["detect_time"] else ("Não" if r["has_fault"] else "—"),
        "t_falha (s)":   f"{r['fault_time']:.1f}" if r["fault_time"] else "—",
        "t_alerta (s)":  f"{r['detect_time']:.1f}" if r["detect_time"] else "—",
        "Latência (s)":  f"{r['latency_s']:.2f}" if r["latency_s"] is not None else "—",
        "Falsos alarmes": r["false_alarms"],
    })

summary = pd.DataFrame(rows)

detected     = summary[summary["Detectado"] == "Sim"]
missed       = summary[(summary["Falha real"] == "Sim") & (summary["Detectado"] == "Não")]
no_fault     = summary[summary["Falha real"] == "Não"]

print(f"Falhas detectadas:     {len(detected)}")
print(f"Falhas não detectadas: {len(missed)}")
print(f"Voos normais:          {len(no_fault)}")
if len(detected) > 0:
    lats = [r["latency_s"] for r in all_results if r["latency_s"] is not None]
    print(f"Latência média:        {np.mean(lats):.2f}s")
    print(f"Latência mediana:      {np.median(lats):.2f}s")

summary

## Visualização por voo

Cada gráfico mostra:
- **Painel superior:** falha real (vermelho) vs. alertas do modelo (azul)
- **Painel inferior:** anomaly score — quanto mais negativo, mais anômalo

In [ ]:
def plot_flight(r: dict, ax_top, ax_bot):
    ts, y_true, y_pred, scores = r["timestamps"], r["y_true"], r["y_pred"], r["scores"]

    ax_top.plot(ts, y_true,  color="red",    alpha=0.7, lw=2,   label="Falha real")
    ax_top.plot(ts, y_pred,  color="steelblue", lw=1, ls="--", label="Alerta modelo")
    if r["fault_time"]:
        ax_top.axvline(r["fault_time"],  color="red",   lw=1.5, ls=":", label=f"t_falha={r['fault_time']:.1f}s")
    if r["detect_time"]:
        ax_top.axvline(r["detect_time"], color="blue",  lw=1.5, ls=":", label=f"t_alerta={r['detect_time']:.1f}s")
    ax_top.set_ylabel("Estado")
    ax_top.set_ylim(-0.1, 1.3)
    ax_top.legend(fontsize=7, loc="upper left")
    ax_top.grid(True, alpha=0.3)

    ax_bot.plot(ts, scores, color="purple", lw=0.7)
    ax_bot.axhline(0, color="red", lw=1.2, ls="--", label="Limiar")
    if r["fault_time"]:
        ax_bot.axvline(r["fault_time"],  color="red",  lw=1.5, ls=":")
    if r["detect_time"]:
        ax_bot.axvline(r["detect_time"], color="blue", lw=1.5, ls=":")
    ax_bot.set_ylabel("Anomaly score")
    ax_bot.set_xlabel("Tempo (s)")
    ax_bot.grid(True, alpha=0.3)


for r in all_results:
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
    label = r["flight"].replace("carbonZ_", "")
    if r["latency_s"] is not None:
        titulo = f"{label}  —  latência: {r['latency_s']:.2f}s"
    elif r["has_fault"] and r["detect_time"] is None:
        titulo = f"{label}  —  NÃO DETECTADO"
    else:
        titulo = f"{label}  —  voo normal"
    fig.suptitle(titulo, fontsize=10, fontweight="bold")
    plot_flight(r, ax_top, ax_bot)
    plt.tight_layout()
    plt.show()

## Distribuição de latências (voos com falha detectada)

In [ ]:
latencias = [(r["flight"].replace("carbonZ_", ""), r["latency_s"])
             for r in all_results if r["latency_s"] is not None]

if latencias:
    nomes, vals = zip(*sorted(latencias, key=lambda x: x[1]))
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(range(len(vals)), vals, color="steelblue", alpha=0.8)
    ax.axvline(np.mean(vals),   color="red",    lw=2, ls="--", label=f"Média: {np.mean(vals):.2f}s")
    ax.axvline(np.median(vals), color="orange", lw=2, ls="--", label=f"Mediana: {np.median(vals):.2f}s")
    ax.set_yticks(range(len(nomes)))
    ax.set_yticklabels([n[:55] for n in nomes], fontsize=8)
    ax.set_xlabel("Latência de detecção (s)")
    ax.set_title("Latência de detecção por voo")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma falha detectada — verifique o parâmetro contamination.")

## Matriz de Confusão — Todos os voos (nível de janela)

Agrega todos os rótulos reais (`y_true`) e predições (`y_pred`) de todos os voos,
avaliando o desempenho janela a janela.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Nível de voo: falha real vs modelo alertou em algum momento
flight_y_true = np.array([int(r["has_fault"])                    for r in all_results])
flight_y_pred = np.array([int(r["y_pred"].any())                 for r in all_results])

cm = confusion_matrix(flight_y_true, flight_y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["Sem falha", "Com falha"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Matriz de Confusão — nível de voo\n(alerta em qualquer momento)")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
recall    = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
print(f"Precisão: {precision:.3f}  |  Recall: {recall:.3f}")

## Matriz de Confusão — Detecção antecipada

Filtra apenas os voos que **tiveram falha real** e nos quais o modelo emitiu
ao menos um alerta **antes** do início da falha real (`y_pred == 1` com índice
anterior ao primeiro índice com `y_true == 1`).

A matriz avalia o desempenho janela a janela somente nesses voos.

In [ ]:
def predicted_early(r: dict) -> bool:
    """Retorna True se o modelo alertou ANTES do início da falha real."""
    if not r["has_fault"]:
        return False
    real_idx = int(np.where(r["y_true"] == 1)[0][0])
    return bool(np.any((r["y_pred"] == 1) & (np.arange(len(r["y_pred"])) < real_idx)))

# Nível de voo: falha real vs modelo alertou ANTES da falha
early_y_true = np.array([int(r["has_fault"])   for r in all_results])
early_y_pred = np.array([int(predicted_early(r)) for r in all_results])

# Resumo por voo
print("Voos com falha — detecção antecipada:")
for r in all_results:
    if not r["has_fault"]:
        continue
    real_idx    = int(np.where(r["y_true"] == 1)[0][0])
    early       = predicted_early(r)
    if early:
        first_alert = int(np.where(
            (r["y_pred"] == 1) & (np.arange(len(r["y_pred"])) < real_idx)
        )[0][0])
        antecipacao = float(r["timestamps"][real_idx] - r["timestamps"][first_alert])
        tag = f"ANTECIPADO em {antecipacao:.2f}s"
    else:
        tag = "não detectado antecipadamente"
    print(f"  {r['flight'].replace('carbonZ_', ''):<60}  {tag}")

cm_early = confusion_matrix(early_y_true, early_y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm_early, display_labels=["Sem falha", "Com falha"])
disp.plot(ax=ax, colorbar=False, cmap="Oranges")
ax.set_title("Matriz de Confusão — nível de voo\n(alerta antes da falha real)")
plt.tight_layout()
plt.show()

tn2, fp2, fn2, tp2 = cm_early.ravel()
print(f"TN={tn2}  FP={fp2}  FN={fn2}  TP={tp2}")
precision2 = tp2 / (tp2 + fp2) if (tp2 + fp2) > 0 else float("nan")
recall2    = tp2 / (tp2 + fn2) if (tp2 + fn2) > 0 else float("nan")
print(f"Precisão: {precision2:.3f}  |  Recall: {recall2:.3f}")